# Reproduce the benchmarks

This notebook reruns the paper's benchmarks. Each run trains with one config, then tests
it. It calls `train.main`, the same entry point as the command line.

The configs and the training code are in [`../integrations/`](../integrations/). A run takes
from 40 minutes (ResNet-18 on CIFAR-10) to a day (WideResNet, 200 epochs) on one GPU.

In [ ]:
import sys

import pandas as pd

FOLDER = "../integrations/torch_uncertainty"    # ../integrations/torch_uncertainty_oodeval for the ViTs
sys.path.append(FOLDER)
from train import main

DATA = "./data"       # CIFAR-10, CIFAR-100 and SVHN are downloaded here
SINGLE = None         # trained single models for the fine-tune configs, as SINGLE/cifar10/resnet18.ckpt
METRICS = {"test/cls/Acc": "Acc", "test/cls/NLL": "NLL", "test/cal/ECE": "ECE",
           "ood/AUROC": "AUROC", "ood/FPR95": "FPR95"}


def run(config, *args):
    """Train and test one config. Extra command line arguments go in `args`."""
    if config.endswith("_finetune"):
        args = ("--model.checkpoint", f"{SINGLE}/{config.removesuffix('_finetune')}.ckpt", *args)
    cli = main(["fit", "--config", f"{FOLDER}/configs/{config}.yaml", "--data.root", DATA, *args])
    metrics = {key: value.item() for key, value in cli.trainer.callback_metrics.items()}
    result = {name: metrics[key] for key, name in METRICS.items() if key in metrics}
    for group in ["near", "far"]:                      # the ViT's OpenOOD sets
        aurocs = [v for k, v in metrics.items() if k.startswith(f"ood_{group}_") and k.endswith("_AUROC")]
        if aurocs:
            result[f"{group} AUROC"] = sum(aurocs) / len(aurocs)
    return result


results = {}

## One run

ResNet-18 on CIFAR-10, trained from scratch.

In [ ]:
results["cifar10/resnet18"] = run("cifar10/resnet18")
results["cifar10/resnet18"]

## All runs

The fine-tune configs start from a trained single model, one per dataset and backbone. Train them
with TorchUncertainty's standard configs, as the integration's README describes. Save them as
`SINGLE/<dataset>/<backbone>.ckpt`, and set `SINGLE`. Without it, only the from-scratch configs
run.

In [ ]:
for dataset in ["cifar10", "cifar100"]:
    for backbone in ["resnet18", "resnet50", "wideresnet28x10"]:
        for variant in ["", "_finetune"] if SINGLE else [""]:
            config = f"{dataset}/{backbone}{variant}"
            if config not in results:
                results[config] = run(config)

## ViT

ViT-B/16 on CIFAR-100 and ImageNet. This section needs another environment. Install
`../integrations/torch_uncertainty_oodeval/requirements.txt`, set `FOLDER` in the first cell to
that folder, and run only the first cell and this section.

ImageNet is not downloaded. Set `IMAGENET` to a folder with `train/`, `val/` and the devkit.
Batch 256 fits an 80 GB GPU.

In [ ]:
IMAGENET = "./data/imagenet"

results["cifar100/vit_b16"] = run("cifar100/vit_b16")
results["imagenet/vit_b16"] = run("imagenet/vit_b16", "--data.root", IMAGENET)

## Compare with the paper

In [ ]:
PAPER = {  # Acc, NLL, ECE from Tables 1, 3, 5 and 6
    "cifar10/resnet18": (93.9, 0.185, 0.7), "cifar10/resnet18_finetune": (94.1, 0.180, 0.85),
    "cifar10/resnet50": (95.4, 0.156, 1.2), "cifar10/resnet50_finetune": (94.3, 0.178, 0.99),
    "cifar10/wideresnet28x10": (96.0, 0.127, 0.8), "cifar10/wideresnet28x10_finetune": (95.1, 0.157, 0.79),
    "cifar100/resnet18": (77.8, 0.826, 1.6), "cifar100/resnet18_finetune": (76.9, 0.831, 2.09),
    "cifar100/resnet50": (80.8, 0.701, 1.6), "cifar100/resnet50_finetune": (77.5, 0.788, 1.94),
    "cifar100/wideresnet28x10": (84.4, 0.584, 5.1), "cifar100/wideresnet28x10_finetune": (80.2, 0.663, 1.31),
    "cifar100/vit_b16": (93.12, 0.247, 1.45), "imagenet/vit_b16": (82.32, 0.655, 1.10),
}
rows = {config: {"Acc": 100 * r["Acc"], "paper Acc": PAPER[config][0],
                 "NLL": r["NLL"], "paper NLL": PAPER[config][1],
                 "ECE": 100 * r["ECE"], "paper ECE": PAPER[config][2]}
        for config, r in results.items()}
pd.DataFrame(rows).T.round(3)